# Webscraping and APIs {#sec-webscraping-and-apis}

## Introduction

This chapter will show you how to work with online data that is either obtained from webpages via webscraping or more directly over the internet via an API. An important principle is always to use an API if one is available as this is designed to pass information directly into your Python session and will save you a lot of effort.


In [1]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Ethical and legal considerations

As we've already said, you should use an API whenever you can. But, for the cases when you can't, what are the rules of the game when it comes to webscraping? First, we need to talk about whether it’s legal and ethical for you to do so. Overall, the situation is complicated with regards to both of these.

Legalities depend a lot on where you live. However, as a general principle, if the data is public, non-personal, non-commercial, and factual, you’re likely to be ok. These three factors are important because they’re connected to the site’s terms and conditions, personally identifiable information, and copyright, as we’ll discuss below.

If the data isn’t public, non-personal, or factual or you’re scraping the data specifically to make money with it, or you're scraping something that is only available commercially, you’ll need to talk to a lawyer. In any case, you should be respectful of the resources of the server hosting the pages you are scraping. Most importantly, this means that if you’re scraping many pages, you should make sure to wait a little between each request. [Tenacity](https://tenacity.readthedocs.io/en/latest/) is an excellent package for this—it has a function that pauses between attempts to do something (as well as a bunch of other useful features).

If you look closely, you’ll find many websites include a “terms and conditions” or “terms of service” link somewhere on the page, and if you read that page closely you’ll often discover that the site specifically prohibits web scraping. These pages tend to be a legal land grab where companies make very broad claims. It’s polite to respect these terms of service where possible, but take any claims with a grain of salt.

US courts have generally found that simply putting the terms of service in the footer of the website isn’t sufficient for you to be bound by them, e.g., [HiQ Labs v. LinkedIn](https://en.wikipedia.org/wiki/HiQ_Labs_v._LinkedIn). Generally, to be bound to the terms of service, you must have taken some explicit action like creating an account or checking a box. This is why whether or not the data is **public** is important; if you don’t need an account to access them, it is unlikely that you are bound to the terms of service. Note, however, the situation is rather different in Europe where courts have found that terms of service are enforceable even if you don’t explicitly agree to them.

Even if the data is public, you should be extremely careful about scraping personally identifiable information like names, email addresses, phone numbers, dates of birth, etc. Europe has strict laws about the collection or storage of such data (known as GDPR), and regardless of where you live you’re likely to be entering an ethical quagmire. For example, in 2016, a group of researchers scraped public profile information (e.g., usernames, age, gender, location, etc.) about 70,000 people on the dating site OkCupid and they publicly released these data without any attempts at anonymisation. While the researchers felt that there was nothing wrong with this since the data were already public, this work was widely condemned due to ethics concerns around identifiability of users whose information was released in the dataset. If your work involves scraping personally identifiable information, we strongly recommend reading about the OkCupid study3 as well as similar studies with questionable research ethics involving the acquisition and release of personally identifiable information.

Finally, you also need to worry about copyright law. Copyright law is complicated. US law describes exactly what’s protected: “[…] original works of authorship fixed in any tangible medium of expression, […]”. It then goes on to describe specific categories that it applies like literary works, musical works, motion pictures and more. Notably absent from copyright protection are data. This means that as long as you limit your scraping to facts, copyright protection does not apply. (But note that Europe has a separate “sui generis” right that protects databases.)

As a brief example, in the US, lists of ingredients and instructions are not copyrightable, so copyright can not be used to protect a recipe. But if that list of recipes is accompanied by substantial novel literary content, that is copyrightable. This is why when you’re looking for a recipe on the internet there’s always so much content beforehand.

If you do need to scrape original content (like text or images), you may still be protected under the doctrine of fair use. Fair use is not a hard and fast rule, but weighs up a number of factors. It’s more likely to apply if you are collecting the data for research or non-commercial purposes and if you limit what you scrape to just what you need.


### Prerequisites

You will need to install the **pandas** and **polars** package for this chapter. We'll use **seaborn** too, which you should already have installed. You will also need to install the **beautifulsoup**, **pandas-datareader**, and **wbgapi** packages in your terminal using `uv add beautifulsoup4`, and `uv add wbgapi` respectively. We'll also use two built-in packages, **textwrap** and **requests**.

To kick off, let's import some of the packages we need (it's always good practice to import the packages you need at the top of a script or notebook).


In [2]:
import textwrap

import lets_plot as lp
import pandas as pd
import polars as pl
import requests
from bs4 import BeautifulSoup

## Extracting Data from Files on the Internet using **polars**

It's easy to read data from the internet once you have the url and file type. Here, for instance, is an example that reads in the 'storms' dataset, which is stored as a CSV file in a URL (we'll only grab the first 10 rows):


In [3]:
pl.read_csv(
    "https://vincentarelbundock.github.io/Rdatasets/csv/dplyr/storms.csv", n_rows=10
)

rownames,name,year,month,day,hour,lat,long,status,category,wind,pressure,tropicalstorm_force_diameter,hurricane_force_diameter
i64,str,i64,i64,i64,i64,f64,f64,str,i64,i64,i64,str,str
1,"""Amy""",1975,6,27,0,27.5,-79.0,"""tropical depression""",null,25,1013,null,null
2,"""Amy""",1975,6,27,6,28.5,-79.0,"""tropical depression""",null,25,1013,null,null
3,"""Amy""",1975,6,27,12,29.5,-79.0,"""tropical depression""",null,25,1013,null,null
4,"""Amy""",1975,6,27,18,30.5,-79.0,"""tropical depression""",null,25,1013,null,null
5,"""Amy""",1975,6,28,0,31.5,-78.8,"""tropical depression""",null,25,1012,null,null
6,"""Amy""",1975,6,28,6,32.4,-78.7,"""tropical depression""",null,25,1012,null,null
7,"""Amy""",1975,6,28,12,33.3,-78.0,"""tropical depression""",null,25,1011,null,null
8,"""Amy""",1975,6,28,18,34.0,-77.0,"""tropical depression""",null,30,1006,null,null
9,"""Amy""",1975,6,29,0,34.4,-75.8,"""tropical storm""",null,35,1004,null,null


## Obtaining data using APIs

Using an API (application programming interface) is another way to draw down information from the interweb. Their just a way for one tool, say Python, to speak to another tool, say a server, and usefully exchange information. The classic use case would be to post a request for data that fits a certain query via an API and to get a download of that data back in return. (You should always preferentially use an API over webscraping a site.)

Because they are designed to work with any tool, you don't actually need a programming language to interact with an API, it's just a _lot_ easier if you do.

::: {.callout-note}
An API key is needed in order to access some APIs. Sometimes all you need to do is register with site, in other cases you may have to pay for access.
:::

To see this, let's directly use an API to get some time series data. We will make the call out to the internet using the **requests** package.

An API has an 'endpoint', the base url, and then a URL that encodes the question. Let's see an example with the ONS API for which the endpoint is "https://api.beta.ons.gov.uk/v1/". The rest of the API has the form 'data?uri=' and then the long ID of both the timeseries (jp9z) and then the dataset (LMS), which is vacancies in the UK services sector.

The data that are returned by APIs are typically in JSON format, which looks a lot like a nested Python dictionary and its entries can be accessed in the same way--this is what is happening when getting the series' title in the example below. JSON is not good for analysis, so we'll use **polars** to put the data into shape.


In [4]:
url = "https://api.beta.ons.gov.uk/v1/data?uri=/employmentandlabourmarket/peopleinwork/employmentandemployeetypes/timeseries/jp9z/lms/previous/v108"

# Get the data from the ONS API:
json_data = requests.get(url).json()

title = json_data["description"]["title"]

# Convert dates using string operations
df = (
    pl.DataFrame(json_data["months"])
    .with_columns(
        [
            # Add day to make it a valid date string
            (pl.col("date") + "-01").str.to_date(format="%Y %b-%d").alias("date"),
            pl.col("value").cast(pl.Float64).alias("value"),
        ]
    )
    .drop_nulls("date")
    .sort("date")
)


# Initialize the library
lp.LetsPlot.setup_html()

# Create plot using the alias
chart = (
    lp.ggplot(df, lp.aes(x="date", y="value"))
    + lp.geom_line(size=2.0, color="steelblue")
    + lp.ggtitle(title)
    + lp.ylim(0, df["value"].max() * 1.2)
    + lp.theme_classic()
)

chart

We've talked about _reading_ APIs. You can also create your own to serve up data, models, whatever you like! This is an advanced topic and we won't cover it; but if you do need to, the simplest way is to use [Fast API](https://fastapi.tiangolo.com/). You can find some short video tutorials for Fast API [here](https://calmcode.io/fastapi/hello-world.html).

### Accessing World Bank Data with wbgapi


While APIs can be accessed directly using tools like requests, some specialized libraries make working with structured datasets much easier. One such example is wbgapi, which provides a convenient interface for accessing World Bank data.

Let’s look at an example using World Bank data on CO₂-equivalent emissions per capita:


In [5]:
# World Bank CO2 equivalent emissions (metric tons per capita)
# https://data.worldbank.org/indicator/EN.GHG.ALL.PC.CE.AR5
# country and region codes at http://api.worldbank.org/v2/country
import wbgapi as wb

indicator_code = "EN.GHG.ALL.PC.CE.AR5"

df = (
    pl.from_pandas(
        wb.data.DataFrame(
            indicator_code,
            ["USA", "CHN", "IND", "EAS", "ECS"],
            time=range(2019, 2020),
            labels=True,
        ).reset_index()
    )
    .rename({"Country": "country"})
    .with_columns(pl.col("country").map_elements(lambda x: textwrap.fill(x, 10)))
    .sort(indicator_code, descending=True)
)

df.head()

/var/folders/j1/0d3dmg7d6gnbc_zwrh8t9gcw0000gn/T/ipykernel_61971/1219974464.py:18: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  .with_columns(pl.col("country").map_elements(lambda x: textwrap.fill(x, 10)))


economy,country,EN.GHG.ALL.PC.CE.AR5
str,str,f64
"""USA""","""United States""",18.921098
"""CHN""","""China""",9.982534
"""ECS""","""Europe & Central Asia""",9.171446
"""EAS""","""East Asia & Pacific""",8.435052
"""IND""","""India""",2.621464


In [6]:
lp.LetsPlot.setup_html()

country_order = df["country"].to_list()

plot = (
    lp.ggplot(df, lp.aes(x="country", y=indicator_code))
    + lp.geom_bar(lp.aes(fill="country"), color="black", alpha=0.8, stat="identity")
    + lp.scale_x_discrete(limits=country_order)
    + lp.scale_fill_discrete()
    + lp.theme_minimal()
    + lp.theme(legend_position="none")
    + lp.ggsize(600, 400)
    + lp.labs(
        subtitle="Greenhouse gases (CO2-equivalent metric tons per capita, 2019)",
        title="The USA leads the world on per-capita emissions",
        y="",
    )
)

plot.show()

### The Eurostat SDMX API

Sometimes it’s convenient to use APIs directly. The Eurostat API provides access to a massive repository of European statistical data using the SDMX (Statistical Data and Metadata eXchange) standard. While Eurostat offers multiple formats, using the SDMX-ML (XML) format via the sdmx1 library allows us to pull structured data into the Python ecosystem with high precision.

Key to using the Eurostat API is understanding the Data Structure Definition (DSD). Every dataset is essentially a multidimensional "cube" where each dimension (like Geography, Unit, or Frequency) has specific codes.

To find the exact codes you need:

The Data Browser: Browse the Eurostat Data Navigation Tree. Once you find a table (e.g., "HICP - monthly data"), the "Dataset Code" (like prc_hicp_manr) is shown in brackets.

Positional Keys: Eurostat's REST API expects a "key string" where codes are placed in a specific order separated by dots (e.g., Freq.Unit.Item.Geo). If you know the order, you can "slice" the data cube directly.

Let’s see an example of this in action. We want to see the Harmonised Index of Consumer Prices (HICP)—specifically the annual rate of change for all items—for Germany and France. We will use the resource prc_hicp_manr, requesting Monthly frequency (M), the Annual Rate of Change unit (RCH_A), and the "All-items" classification (CP00).


```python
import sdmx
import polars as pl

# Tell sdmx we want ESTAT data
client = sdmx.Client('ESTAT')

# 2. Build the URL-style positional key
# Format: [Freq].[Unit].[Coicop].[Geo]
# We use '+' to join multiple countries (DE and FR)
resource_id = 'prc_hicp_manr'
key_string = 'M.RCH_A.CP00.DE+FR'

# 3. Fetch the data directly
# 'startPeriod' limits the timeline to recent data
response = client.data(
    resource_id=resource_id,
    key=key_string,
    params={'startPeriod': '2024-01'}
)

# 4. Convert the SDMX-ML response to a Polars DataFrame
# We bridge through Pandas as sdmx1 is optimized for it
df_pd = sdmx.to_pandas(response).to_frame(name='value').reset_index()
df = pl.from_pandas(df_pd)

print(df.head())
```


|     | TIME_PERIOD | geo | unit  | freq | coicop | value |
| --: | ----------: | :-- | :---- | :--- | :----- | ----: |
|   0 |     2024-01 | DE  | RCH_A | M    | CP00   |   3.1 |
|   1 |     2024-02 | DE  | RCH_A | M    | CP00   |   2.7 |
|   2 |     2024-03 | DE  | RCH_A | M    | CP00   |   2.3 |
|   3 |     2024-04 | DE  | RCH_A | M    | CP00   |   2.4 |
|   4 |     2024-05 | DE  | RCH_A | M    | CP00   |   2.8 |


Great that worked! We have data in a nice tidy format.


### Other Useful APIs

- There is a regularly updated list of APIs over at this [public APIs repo on github](https://github.com/public-apis/public-apis). It doesn't have an economics section (yet), but it has a LOT of other APIs.
- Berkeley Library maintains a [list of economics APIs](https://guides.lib.berkeley.edu/c.php?g=4395&p=7995952) that is well worth looking through.
- [NASDAQ Data Link](https://docs.data.nasdaq.com/), which has a great deal of [financial data](https://docs.data.nasdaq.com/docs/data-organization).
- [DBnomics](https://db.nomics.world/): publicly-available economic data provided by national and international statistical institutions, but also by researchers and private companies.


## Webscraping

Webscraping is a way of grabbing information from the internet that was intended to be displayed in a browser. But it should only be used as a last resort, and only then when permitted by the terms and conditions of a website.

If you're getting data from the internet, it's much better to use an API whenever you can: grabbing information in a structure way is _exactly_ why APIs exist. APIs should also be more stable than websites, which may change frequently. Typically, if an organisation is happy for you to grab their data, they will have made an API expressly for that purpose. It's pretty rare that there's a major website which _does_ permit webscraping but which doesn't have an API; for these websites, if they don't have an API, chances scraping is against their terms and conditions. Those terms and conditions may be enforceable by law (different rules in different countries here, and you really need legal advice if it's not unambiguous as to whether you can scrape or not.)

There are other reasons why webscraping is not so good; for example, if you need a back-run then it might be offered through an API but not shown on the webpage. (Or it might not be available at all, in which case it's best to get in touch with the organisation or check out WaybackMachine in case they took snapshots).

So this book is pretty down on webscraping as there's almost always a better solution. However, there are times when it is useful.

If you do find yourself in a scraping situation, be really sure to check that's legally allowed and also that you are not violating the website's `robots.txt` rules: this is a special file on almost every website that sets out what's fair play to crawl (conditional on legality) and what robots should not go poking around in.

In Python, you are spoiled for choice when it comes to webscraping. There are five very strong libraries that cover a real range of user styles and needs: **requests**, **lxml**, **beautifulsoup**, **selenium**, and \*scrapy\*\*.

For quick and simple webscraping, my usual combo would **requests**, which does little more than go and grab the HTML of a webpage, and **beautifulsoup**, which then helps you to navigate the structure of the page and pull out what you're actually interested in. For dynamic webpages that use javascript rather than just HTML, you'll need **selenium**. To scale up and hit thousands of webpages in an efficient way, you might try **scrapy**, which can work with the other tools and handle multiple sessions, and all other kinds of bells and whistles... it's actually a "web scraping framework".

It's always helpful to see coding in practice, so that's what we'll do now, but note that we'll be skipping over a lot of important detail such as user agents, being 'polite' with your scraping requests, being efficient with caching and crawling.

In lieu of a better example, let's scrape the research page of [http://aeturrell.com/](http://aeturrell.com/)


In [7]:
url = "http://aeturrell.com/research"
page = requests.get(url)
page.text[:300]

'<!DOCTYPE html>\n<html xmlns="http://www.w3.org/1999/xhtml" lang="en" xml:lang="en"><head>\n\n<meta charset="utf-8">\n<meta name="generator" content="quarto-1.9.32">\n\n<meta name="viewport" content="width=device-width, initial-scale=1.0, user-scalable=yes">\n\n<meta name="author" content="Arthur Turrell">\n'

Okay, what just happened? We asked requests to grab the HTML of the webpage and then printed the first 300 characters of the text that it found.

Let's now parse this into something humans can read (or can read more easily) using beautifulsoup:


In [8]:
soup = BeautifulSoup(page.text, "html.parser")
print(soup.prettify()[60000:60500])

>
         <div class="project-categories">
          <div class="project-category">
           <a href="#category=data science">
            data science
           </a>
          </div>
          <div class="project-category">
           <a href="#category=public policy">
            public policy
           </a>
          </div>
          <div class="project-category">
           <a href="#category=complexity science">
            complexity science
           </a>
          </div>
         <


Now we see more structure of the page and even some _HTML tags_ such as 'title' and 'link'. Now we come to the data extraction part: say we want to pull out every paragraph of text, we can use beautifulsoup to skim down the HTML structure and pull out only those parts with the paragraph tag ('p').


In [9]:
# Get all paragraphs
all_paras = soup.find_all("p")
# Just show one of the paras
all_paras[1]

<p>Blundell, Jack, Emma Duchini, Stefania Simion, and Arthur Turrell. "Pay transparency and gender equality." <i>American Economic Journal: Economic Policy</i> (2024). doi: <a href="https://www.aeaweb.org/articles?id=10.1257/pol.20220766&amp;from=f"><code>10.1257/pol.20220766</code></a></p>

Although this paragraph isn't too bad, you can make this more readable by stripping out HTML tags altogether with the `.text` method:


In [10]:
all_paras[1].text

'Blundell, Jack, Emma Duchini, Stefania Simion, and Arthur Turrell. "Pay transparency and gender equality." American Economic Journal: Economic Policy (2024). doi: 10.1257/pol.20220766'

Now let's say we didn't care about most of the page, we _only_ wanted to get hold of the names of projects. For this we need to identify the tag type of the element we're interested in, in this case 'div', and it's class type, in this case "project-name". We do it like this (and show nice text in the process):


In [11]:
projects = soup.find_all("div", class_="project-content listing-pub-info")
projects = [x.text.strip() for x in projects]
projects

['Blundell, Jack, Emma Duchini, Stefania Simion, and Arthur Turrell. "Pay transparency and gender equality." American Economic Journal: Economic Policy (2024). doi: 10.1257/pol.20220766',
 'Botta, Federico, Robin Lovelace, Laura Gilbert, and Arthur Turrell. "Packaging code and data for reproducible research: A case study of journey time statistics." Environment and Planning B: Urban Analytics and City Science (2024): 23998083241267331. doi: 10.1177/23998083241267331',
 'Kalamara, Eleni, Arthur Turrell, Chris Redl, George Kapetanios, and Sujit Kapadia. "Making text count: economic forecasting using newspaper text." Journal of Applied Econometrics 37, no. 5 (2022): 896-919. doi: 10.1002/jae.2907',
 'Turrell, A., Speigner, B., Copple, D., Djumalieva, J. and Thurgood, J., 2021. Is the UK’s productivity puzzle mostly driven by occupational mismatch? An analysis using big data on job vacancies. Labour Economics, 71, p.102013. doi: 10.1016/j.labeco.2021.102013',
 'Haldane, Andrew G., and Arth

Hooray! We managed to get the information we wanted: all we needed to know was the right tags. A good tip for finding the tags of the info you want is to look at in your browser (eg Google Chrome) and then right-click on the bit you're interested in, then hit 'Inspect'. This will show you the HTML element of the bit of the page you clicked on.

That's almost it for this very, very brief introduction to webscraping. We'll just see one more thing: how to iterate over multiple pages.

Imagine we had a root webpage such as "www.codingforeconomists.com" which had subpages such as "www.codingforeconomists.com/page=1", "www.codingforeconomists.com/page=2", and so on. One need only iterate create the HTML strings to pass into a function that scrapes each one and return the relevant data, eg for the first 50 pages, and with a function called `scraper()`, one might run

```
start, stop = 0, 50
root_url = "www.codingforeconomists.com/page="
info_on_pages = [scraper(root_url + str(i)) for i in range(start, stop)]
```

That's all we'll cover here but remember we've barely _scraped_ the surface of this big, complex topic. If you want to read about an application, it's hard not to recommend the paper on webscraping that has undoubtedly change the world the most, and very likely has affected your own life in numerous ways: ["The PageRank Citation Ranking: Bringing Order to the Web"](http://ilpubs.stanford.edu:8090/422/) by Page, Brin, Motwani and Winograd. For a more in-depth example of webscraping, check out realpython's [tutorial](https://realpython.com/python-web-scraping-practical-introduction/).


### Webscraping Tables

There are times when you don't need to scrape an entire webpage; you simply want the structured data from a specific table. While Polars is a high-performance data engine, it focuses on strict data formats (like Parquet or CSV) and does not natively include an HTML parser. However, we can easily bridge this gap by using Pandas to fetch the table and then converting it into a Polars DataFrame.

We will read data from 'https://webscraper.io/test-sites/tables' using `pd.read_html()`. This function scans the webpage and returns a list of all tables it finds as DataFrames. To target a specific table, we use the match= keyword argument with text that uniquely appears in the table we want—in this case, "First Name".

Once captured, we convert the result to Polars using pl.from_pandas() to take advantage of Polars' superior query performance and expression API.


In [12]:
import polars as pl

pd_list = pd.read_html("https://webscraper.io/test-sites/tables", match="First Name")
# Retrieve first entry from list of data frames
df = pl.from_pandas(pd_list[0])

print(df.head())

shape: (3, 4)
┌─────┬────────────┬───────────┬──────────┐
│ #   ┆ First Name ┆ Last Name ┆ Username │
│ --- ┆ ---        ┆ ---       ┆ ---      │
│ i64 ┆ str        ┆ str       ┆ str      │
╞═════╪════════════╪═══════════╪══════════╡
│ 1   ┆ Mark       ┆ Otto      ┆ @mdo     │
│ 2   ┆ Jacob      ┆ Thornton  ┆ @fat     │
│ 3   ┆ Larry      ┆ the Bird  ┆ @twitter │
└─────┴────────────┴───────────┴──────────┘


This gives us the table neatly loaded into a **polars** data frame ready for further use.

If you get a '403' error, it means that the website has blocked **pandas** because it can see that you are engaged in web scraping. This is because some people web scrape irresponsibly, or because websites have provided other, preferred ways for you to obtain the data, eg via a download of the whole thing (think Wikipedia) or through an API. (If you really need to, [you can often get around the 403 error](https://stackoverflow.com/questions/43590153/http-error-403-forbidden-when-reading-html) though.)
